In [1]:
import os
import glob
import pandas as pd
import torch
import torchaudio
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader,random_split
import librosa
import numpy as np
from tqdm import tqdm
from natsort import natsorted
import random
import soundfile as sf
from torchmetrics.audio.pesq import PerceptualEvaluationSpeechQuality
from torchmetrics.audio.stoi import ShortTimeObjectiveIntelligibility


In [2]:
import kagglehub
noise_audio= kagglehub.dataset_download('javohirtoshqorgonov/noise-audio-data')
noise_audio

'/teamspace/studios/this_studio/.cache/kagglehub/datasets/javohirtoshqorgonov/noise-audio-data/versions/1'

In [3]:
import kagglehub
muhmagdy_valentini_noisy_path= kagglehub.dataset_download('muhmagdy/valentini-noisy')
muhmagdy_valentini_noisy_path

'/teamspace/studios/this_studio/.cache/kagglehub/datasets/muhmagdy/valentini-noisy/versions/3'

In [4]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [5]:
def set_seed(seed=14):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(14)

In [ ]:
SR=16000
MAX_LEN=80000 # 5 seconds
BATCH_SIZE=16

N_FFT=512
HOP_LENGTH=160  
CENTER=False
FREQ_BINS=256

DETACH_EVERY=20
REAL_TIME_CHUNK_FRAMES=4
TRAIN_CHUNK_FRAMES=16
VAL_CHUNK_FRAMES=16
train_chunk_frame_choices=[8,16]

ENABLE_TRANSIENT_AUG=True
TRANSIENT_PROB=0.15
TRANSIENT_MAX_EVENTS=1
TRANSIENT_SNR_RANGE=(-3,20)
TRANSIENT_LOSS_WEIGHT=0.4

NUM_WORKERS=4
PIN_MEMORY=torch.cuda.is_available()
USE_AMP=torch.cuda.is_available()

RUN_PESQ_STOI_EVERY_EPOCH=False
PESQ_STOI_EVERY_N_EPOCHS=2
MAX_METRIC_BATCHES=10

In [7]:
import pandas as pd

df = pd.read_csv("esc50.csv")

print("Number of unique categories:", df["category"].nunique())
print()


df["category"].unique()

Number of unique categories: 50



array(['dog', 'chirping_birds', 'vacuum_cleaner', 'thunderstorm',
       'door_wood_knock', 'can_opening', 'crow', 'clapping', 'fireworks',
       'chainsaw', 'airplane', 'mouse_click', 'pouring_water', 'train',
       'sheep', 'water_drops', 'church_bells', 'clock_alarm',
       'keyboard_typing', 'wind', 'footsteps', 'frog', 'cow',
       'brushing_teeth', 'car_horn', 'crackling_fire', 'helicopter',
       'drinking_sipping', 'rain', 'insects', 'laughing', 'hen', 'engine',
       'breathing', 'crying_baby', 'hand_saw', 'coughing',
       'glass_breaking', 'snoring', 'toilet_flush', 'pig',
       'washing_machine', 'clock_tick', 'sneezing', 'rooster',
       'sea_waves', 'siren', 'cat', 'door_wood_creaks', 'crickets'],
      dtype=object)

In [8]:
ESC50_CONTINUOUS = {
    "rain",
    "sea_waves",
    "wind",
    "thunderstorm",
    "water_drops",
    "pouring_water",
    "crackling_fire",
    "engine",
    "vacuum_cleaner",
    "washing_machine",
    "train",
    "airplane",
    "helicopter"
}

ESC50_TRANSIENT = {
    "door_wood_knock",
    "glass_breaking",
    "car_horn",
    "siren",
    "clock_alarm",
    "mouse_click",
    "keyboard_typing",
    "can_opening",
    "church_bells",
    "clapping",
    "fireworks"
}

In [9]:
class SmartNoiseDataset(Dataset):
    #  Lifecycle                                                          
    def __init__(
        self,
        clean_paths,
        demand_noise_path,
        esc50_audio_path,
        esc50_csv_path,
        sr=16000,
        max_len=80000
    ):
        super().__init__()

        self.sr = sr
        self.max_len = max_len

        self.clean_files = []
        for path in clean_paths:
            if os.path.exists(path):
                self.clean_files.extend(self._get_files(path))
        random.shuffle(self.clean_files)

        self.demand_noises = self._get_files(demand_noise_path)

        self.esc50_continuous = []
        self.esc50_transient = []
        self._load_esc50(esc50_audio_path, esc50_csv_path)

        if len(self.clean_files) == 0:
            raise RuntimeError("No clean files found")
        if len(self.demand_noises) == 0:
            raise RuntimeError("No DEMAND noises found")

        print(f"Clean files: {len(self.clean_files)}")
        print(f"DEMAND noises: {len(self.demand_noises)}")
        print(f"ESC50 continuous: {len(self.esc50_continuous)}")
        print(f"ESC50 transient: {len(self.esc50_transient)}")

    def __len__(self):
        return len(self.clean_files)

    def __getitem__(self, idx):
        try:
            clean_path = self.clean_files[idx]
            file_name = os.path.basename(clean_path)
            clean = self._load_speech(clean_path)

            r = random.random()

            if r < 0.05:
                noisy = clean.copy()

            elif r < 0.10:
                noisy = (
                    clean +
                    np.random.normal(0, 0.001, len(clean)).astype(np.float32)
                )

            elif r < 0.60:
                demand = self._load_noise(random.choice(self.demand_noises))
                noisy = self._mix_with_snr(clean, demand, (-2, 25))

            elif r < 0.80:
                n1 = self._load_noise(random.choice(self.demand_noises))
                n2 = self._load_noise(random.choice(self.demand_noises))
                noisy = self._mix_multiple(clean, [n1, n2])

            elif r < 0.90:
                demand = self._load_noise(random.choice(self.demand_noises))
                esc = self._load_noise(random.choice(self.esc50_continuous))
                noisy = self._mix_multiple(clean, [demand, esc])

            else:
                demand = self._load_noise(random.choice(self.demand_noises))
                esc = self._load_noise(random.choice(self.esc50_transient))
                noisy = self._mix_multiple(clean, [demand, esc])

            clean, noisy = self._normalize_pair(clean, noisy)

            gain = random.uniform(0.7, 1.1)
            clean *= gain
            noisy *= gain

            clean = np.clip(clean, -1, 1)
            noisy = np.clip(noisy, -1, 1)

            return (
                torch.from_numpy(noisy).float(),
                torch.from_numpy(clean).float(),
                file_name
            )

        except Exception as e:
            print(f"Error: {e}")
            return self.__getitem__(random.randint(0, len(self) - 1))

    #  I/O helpers                                                         
   
    def _get_files(self, directory):
        files = []
        for ext in ("**/*.wav", "**/*.mp3", "**/*.flac"):
            files.extend(
                glob.glob(os.path.join(directory, ext), recursive=True)
            )
        return files

    def _load_esc50(self, esc50_audio_path, esc50_csv_path):
        df = pd.read_csv(esc50_csv_path)
        for _, row in df.iterrows():
            file_path = os.path.join(esc50_audio_path, row["filename"])
            category = row["category"]
            if category in ESC50_CONTINUOUS:
                self.esc50_continuous.append(file_path)
            elif category in ESC50_TRANSIENT:
                self.esc50_transient.append(file_path)

    def _load_speech(self, path):
        audio, fs = sf.read(path)

        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)

        if fs != self.sr:
            audio = librosa.resample(audio, orig_sr=fs, target_sr=self.sr)

        if len(audio) == 0:
            return np.zeros(self.max_len, dtype=np.float32)

        if len(audio) > self.max_len:
            start = random.randint(0, len(audio) - self.max_len)
            audio = audio[start:start + self.max_len]
        else:
            pad = self.max_len - len(audio)
            audio = np.pad(audio, (0, pad), mode="constant")

        return audio.astype(np.float32)

    def _load_noise(self, path):
        audio, fs = sf.read(path)

        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)

        if fs != self.sr:
            audio = librosa.resample(audio, orig_sr=fs, target_sr=self.sr)

        if len(audio) == 0:
            return np.zeros(self.max_len, dtype=np.float32)

        if len(audio) > self.max_len:
            start = random.randint(0, len(audio) - self.max_len)
            audio = audio[start:start + self.max_len]
        else:
            repeat = int(np.ceil(self.max_len / len(audio)))
            audio = np.tile(audio, repeat)
            audio = audio[:self.max_len]

        return audio.astype(np.float32)

    #  Audio mixing helpers                                                
    
    def RMS(self, audio):
        return np.sqrt(np.mean(audio ** 2) + 1e-7)

    def _mix_with_snr(self, clean, noise, snr_range):
        snr_db = random.uniform(snr_range[0], snr_range[1])
        clean_rms = self.RMS(clean)
        noise_rms = self.RMS(noise)
        snr_linear = 10 ** (snr_db / 20.0)
        factor = clean_rms / (snr_linear * noise_rms + 1e-7)
        return clean + factor * noise

    def _mix_multiple(self, clean, noise_list):
        noisy = clean.copy()
        for noise in noise_list:
            snr = random.uniform(-2, 20)
            noisy = self._mix_with_snr(noisy, noise, (snr, snr))
        return noisy

    def _normalize_pair(self, clean, noisy, target_db=-25):
        rms = self.RMS(clean)
        current_db = 20 * np.log10(rms + 1e-7)
        gain = 10 ** ((target_db - current_db) / 20)
        clean *= gain
        noisy *= gain

        peak = max(np.abs(clean).max(), np.abs(noisy).max())
        if peak > 0.99:
            scale = 0.99 / peak
            clean *= scale
            noisy *= scale

        return clean.astype(np.float32), noisy.astype(np.float32)

In [10]:
clean_train_path_english = os.path.join(muhmagdy_valentini_noisy_path, 'clean_trainset_56spk_wav')
clean_val_path_english = os.path.join(muhmagdy_valentini_noisy_path, 'clean_trainset_28spk_wav')

In [11]:
train_dataset = SmartNoiseDataset(

    clean_paths=[
        r"/teamspace/studios/this_studio/MeetVerse_DataSet/Clear Sound train",
        clean_train_path_english
    ],

    demand_noise_path=r"/teamspace/studios/this_studio/noise_train",
    esc50_audio_path=r"/teamspace/studios/this_studio/.cache/kagglehub/datasets/javohirtoshqorgonov/noise-audio-data/versions/1/ESC-50-master/audio",
    esc50_csv_path=r"/teamspace/studios/this_studio/esc50.csv",

    sr=16000,
    max_len=80000
)

Clean files: 40445
DEMAND noises: 128
ESC50 continuous: 520
ESC50 transient: 440


In [12]:
val_dataset = SmartNoiseDataset(

    clean_paths=[
        r"/teamspace/studios/this_studio/MeetVerse_DataSet/Clear Sound val",
        clean_val_path_english
    ],
    demand_noise_path=r"/teamspace/studios/this_studio/noise_train",
    esc50_audio_path=r"/teamspace/studios/this_studio/.cache/kagglehub/datasets/javohirtoshqorgonov/noise-audio-data/versions/1/ESC-50-master/audio",
    esc50_csv_path=r"/teamspace/studios/this_studio/esc50.csv",

    sr=16000,
    max_len=80000
)

Clean files: 18508
DEMAND noises: 128
ESC50 continuous: 520
ESC50 transient: 440


In [13]:
train_loader=DataLoader(train_dataset,batch_size=16,shuffle=True,num_workers=4,pin_memory=PIN_MEMORY,persistent_workers=True)
val_loader=DataLoader(val_dataset,batch_size=16,shuffle=False,num_workers=4,pin_memory=PIN_MEMORY,persistent_workers=True)

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def make_group_norm(channels,max_groups=8):
    groups=min(channels,max_groups)
    while groups>1 and channels%groups!=0:
        groups-=1
    return nn.GroupNorm(groups,channels)

class SEBlock(nn.Module): # attention layer
    def __init__(self,channels,reduction=8):
        super().__init__()
        self.pool=nn.AdaptiveAvgPool2d(1)
        hidden=max(channels//reduction,4)
        self.fc=nn.Sequential(
            nn.Conv2d(channels,hidden,kernel_size=1,bias=False),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden,channels,kernel_size=1,bias=False),
            nn.Sigmoid()
        )

    def forward(self,x):
        w=self.pool(x)
        w=self.fc(w)
        return w*x

class ResSEBlock(nn.Module):
    def __init__(self,in_c,out_c,stride):
        super().__init__()
        self.act=nn.SiLU(inplace=True)

        self.conv1=nn.Conv2d(in_c,out_c,kernel_size=3,stride=stride,padding=1)
        self.gn1=make_group_norm(out_c) # best for realtime

        self.conv2=nn.Conv2d(out_c,out_c,kernel_size=3,padding=1)
        self.gn2=make_group_norm(out_c)

        self.se=SEBlock(out_c)

        if stride!=1 or in_c!=out_c:
            self.shortcut=nn.Sequential(
                nn.Conv2d(in_c,out_c,kernel_size=1,stride=stride),
                make_group_norm(out_c)
            )
        else :
            self.shortcut=nn.Identity()

    def forward(self,x):
        identity=self.shortcut(x)

        out=self.conv1(x)
        out=self.gn1(out)
        out=self.act(out)

        out=self.conv2(out)
        out=self.gn2(out)

        out=self.se(out)

        out=out+identity
        out=self.act(out)

        return out

def up_Block(in_c,out_c):
    return nn.Sequential(
        nn.Upsample(scale_factor=(2,1),mode='bilinear',align_corners=False),
        nn.Conv2d(in_c,out_c,kernel_size=3,padding=1),
        make_group_norm(out_c),
        nn.SiLU(inplace=True)
    )

class Model (nn.Module):
    def __init__(self, base=48,gru_hidden_sz=256,layers=1,freq_bins=256):
        super().__init__()
        self.base=base
        self.gru_hidden_sz=gru_hidden_sz
        self.layers=layers
        self.freq_bins=freq_bins

        # Encoding
        self.enc1=ResSEBlock(2,self.base,stride=(2,1))
        self.enc2=ResSEBlock(self.base,self.base*2,stride=(2,1))
        self.enc3=ResSEBlock(self.base*2,self.base*4,stride=(2,1))
        self.enc4=ResSEBlock(self.base*4,self.base*8,stride=(2,1))


        # GRU

        self.bottleneck_freq_bins=self.freq_bins//16
        self.gru_in_sz=(self.base*8)*self.bottleneck_freq_bins


        self.gru=nn.GRU(
            input_size=self.gru_in_sz,
            hidden_size=self.gru_hidden_sz,
            num_layers=self.layers,
            batch_first=True,
            bidirectional=False
        )

        self.gru_fc=nn.Sequential(
            nn.Linear(self.gru_hidden_sz,self.gru_in_sz),
            nn.SiLU(inplace=True),
        )

        # DECODE
        self.dec4=up_Block(self.base*16,self.base*4)
        self.dec3=up_Block(self.base*8,self.base*2)
        self.dec2=up_Block(self.base*4,self.base)

        # self.mask_conv=nn.Sequential()-----------
        self.mask_conv=nn.Sequential(
            nn.Upsample(scale_factor=(2,1),mode='bilinear',align_corners=False),
            nn.Conv2d(self.base*2,2,kernel_size=3,padding=1)
        )

        self.sig=nn.Sigmoid()

    @staticmethod
    def match_shape(x,target):
        diff_f=target.size(2)-x.size(2)
        diff_t=target.size(3)-x.size(3)
        if diff_f==0 and diff_t==0:
            return x
        return F.pad(x,(diff_t//2,diff_t-diff_t//2,diff_f//2,diff_f-diff_f//2,))


    def forward(self,x,h=None):
        e1=self.enc1(x)
        e2=self.enc2(e1)
        e3=self.enc3(e2)
        e4=self.enc4(e3)

        b,c,f,t=e4.shape

        # seq -------------------
        seq=e4.permute(0,3,1,2).reshape(b,t,c*f)

        if c*f!=self.gru_in_sz:
            raise ValueError(f"GRU input mismatch: got {c*f}, expected {self.gru_in_sz}")

        gru_out,next_h=self.gru(seq,h)
        gru_out=self.gru_fc(gru_out)
        gru_out=gru_out.reshape(b,t,c,f).permute(0,2,3,1)


        d4=self.dec4(torch.cat([gru_out,e4],dim=1))
        d4=self.match_shape(d4,e3)

        d3=self.dec3(torch.cat([d4,e3],dim=1))
        d3=self.match_shape(d3,e2)

        d2=self.dec2(torch.cat([d3,e2],dim=1))
        d2=self.match_shape(d2,e1)


        mask=self.mask_conv(torch.cat([d2,e1],dim=1))
        mask=self.match_shape(mask,x)

        mask=self.sig(mask)

        return mask,next_h


In [ ]:
model = Model(
    base=48,
    gru_hidden_sz=512,
    layers=1,
    freq_bins=FREQ_BINS
)

x = torch.randn(1, 2, 256, 2)

mask, h = model(x)

print(mask.shape)
print(h.shape)

torch.Size([1, 2, 256, 2])
torch.Size([1, 1, 512])


In [16]:
import torch
import torch.nn.functional as F

class SpectrogramTransform:
    def __init__(self,n_fft=N_FFT,hop_length=HOP_LENGTH,center=CENTER,device='cpu'):

        self.n_fft=n_fft
        self.hop_length=hop_length
        self.center=center
        self.device=device

        self.window=torch.hamming_window(self.n_fft).to(self.device)

    def pad_for_center_false(self,wav):
        if self.center:
            return wav

        total_len=wav.shape[-1]
        if total_len<self.n_fft:
            pad_amount=self.n_fft-total_len
        else :
            remainder=(total_len-self.n_fft)%self.hop_length
            pad_amount=0 if remainder==0  else self.hop_length-remainder

        if pad_amount>0:
            wav=F.pad(wav,(0,pad_amount))

        return wav

    def audio_to_spectrogram(self,wav):

        if self.window.device!=wav.device:
            self.window=self.window.to(wav.device)

        wav = self.pad_for_center_false(wav)
        spec=torch.stft(
            wav,n_fft=self.n_fft,hop_length=self.hop_length,
            window=self.window,return_complex=True,center=self.center
        )
        spec_features=torch.view_as_real(spec).permute(0,3,1,2)
        spec_features=spec_features[:,:,:-1,:]
        return spec_features

    def apply_mask(self,spec_features,mask):
        return spec_features*mask

    def spectrogram_to_audio(self,enhanced_spec,target_num_samples):
        if self.window.device !=enhanced_spec.device:
            self.window=self.window.to(enhanced_spec.device)

        enhanced_spec=F.pad(enhanced_spec,(0,0,0,1))
        enhanced_spec=enhanced_spec.permute(0,2,3,1).contiguous()

        complex_spec=torch.complex(
            enhanced_spec[...,0],
            enhanced_spec[...,1]
        )
        wav=torch.istft(
            complex_spec,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            window=self.window,
            center=self.center,
            length=target_num_samples
        )

        return wav

In [17]:
import torch
import torch.nn.functional as F
def split_spec_chunks(spec,chunk_frames=TRAIN_CHUNK_FRAMES):
    chunks=[]
    original_T=spec.size(-1)
    for start in range(0,original_T,chunk_frames):
        chunk=spec[:,:,:,start:start+chunk_frames]
        if chunk.size(-1)<chunk_frames:
            pad_t=chunk_frames-chunk.size(-1)
            chunk=F.pad(chunk,(0,pad_t))
        chunks.append(chunk)
    return chunks,original_T

def enhance_waveform_streaming(noisy_wav,model,transform,chunk_frames=TRAIN_CHUNK_FRAMES,h=None,detach_every=DETACH_EVERY):
    noisy_spec=transform.audio_to_spectrogram(noisy_wav)
    chunks,original_T=split_spec_chunks(noisy_spec,chunk_frames=chunk_frames)
    enhanced_chunks=[]
    next_h=h

    for i,spec_chunk in enumerate(chunks):
        mask,next_h=model(spec_chunk,next_h)
        enhanced_chunk=transform.apply_mask(spec_chunk,mask)
        enhanced_chunks.append(enhanced_chunk)

        if detach_every is not None and (i+1)%detach_every==0:
            if next_h is not None:
                next_h=next_h.detach()
    enhanced_spec=torch.cat(enhanced_chunks,dim=-1)
    enhanced_spec=enhanced_spec[:,:,:,:original_T]

    enhanced_wav=transform.spectrogram_to_audio(enhanced_spec,target_num_samples=noisy_wav.shape[-1])

    return enhanced_wav,enhanced_spec,next_h

In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiScaleSpectralLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.fft_sizes=[2048,1024,512,256]
        self.hop_sizes=[240,120,60,30]
        self.window_lens=[1200,600,240,120]

        for i,win in enumerate(self.window_lens):
            self.register_buffer(f"Window_{i}",torch.hann_window(win))

    def forward(self,est,ref):
        total_loss=0.0
        for i,(n_fft,hop,win) in enumerate(zip(self.fft_sizes,self.hop_sizes,self.window_lens)):
            window=getattr(self,f'Window_{i}').to(est.device)

            est_stft=torch.stft(est,n_fft=n_fft,hop_length=hop,win_length=win,window=window,return_complex=True,center=True)
            ref_stft=torch.stft(ref,n_fft=n_fft,hop_length=hop,win_length=win,window=window,return_complex=True,center=True)


            est_mag=torch.abs(est_stft).clamp(min=1e-7).pow(0.3)
            ref_mag=torch.abs(ref_stft).clamp(min=1e-7).pow(0.3)

            sc_loss=torch.norm(ref_mag-est_mag,p="fro")/(torch.norm(ref_mag,p="fro")+1e-7)
            mag_loss=F.l1_loss(est_mag,ref_mag)

            log_est=torch.log(est_mag.clamp(min=1e-7))
            log_ref=torch.log(ref_mag.clamp(min=1e-7))
            log_loss=F.l1_loss(log_est,log_ref)

            total_loss+=sc_loss+mag_loss+log_loss
        return total_loss/len(self.fft_sizes)

class MelLoss(nn.Module):
    def __init__(self,sr=SR,n_mels=80,n_fft=N_FFT,hop_length=HOP_LENGTH):
        super().__init__()
        self.mel=torchaudio.transforms.MelSpectrogram(
            sample_rate=sr,n_mels=n_mels,n_fft=n_fft,hop_length=hop_length
        )
    def forward(self,est,ref):
        mel_est=self.mel(est).clamp(min=1e-7).pow(0.3)
        mel_ref=self.mel(ref).clamp(min=1e-7).pow(0.3)
        return F.l1_loss(mel_est,mel_ref)

class TransientWeightedLoss(nn.Module):
    def __init__(self, weight=TRANSIENT_LOSS_WEIGHT):
        super().__init__()
        self.weight=weight
    def forward(self,pred,clean,noisy):
        noise=(noisy-clean).abs()
        env=F.avg_pool1d(
            noise.unsqueeze(1),
            kernel_size=321,
            stride=1,
            padding=160
        ).squeeze(1)

        env=env/(env.mean(dim=-1,keepdim=True)+1e-7)

        weights=torch.clamp(env,1.0,8.0)
        return (weights*(pred-clean).abs()).mean()

class MasterHybridLoss(nn.Module):
    def __init__(self, weight=TRANSIENT_LOSS_WEIGHT):
        super().__init__()
        self.spectral_loss_fn=MultiScaleSpectralLoss()
        self.transient_loss_fn=TransientWeightedLoss(TRANSIENT_LOSS_WEIGHT)
        self.transient_weight=weight
        self.mel_loss_fn=MelLoss()


    def forward(self,pred,target,noisy=None):
        spec_loss=self.spectral_loss_fn(est=pred,ref=target)
        wave_l1=F.l1_loss(pred,target)
        target_zm=target-target.mean(dim=-1,keepdim=True)
        pred_zm=pred-pred.mean(dim=-1,keepdim=True)

        dot=(target_zm*pred_zm).sum(dim=-1,keepdim=True)
        target_energy=(target_zm**2).sum(dim=-1,keepdim=True)+1e-8

        s_target=(dot/target_energy)*target_zm
        e_noise=pred_zm-s_target

        si_sdr=10*torch.log10((s_target**2).sum(dim=-1)
                            /((e_noise**2).sum(dim=-1)+1e-8)+1e-8)
        sisdr_loss=-si_sdr.mean()
        mel_loss=self.mel_loss_fn(pred,target)

        total_loss=(1.4*spec_loss)+(1*sisdr_loss)+(0.25*wave_l1)+(0.35*mel_loss)

        if noisy is not None and self.transient_weight>0:
            trans_loss=self.transient_loss_fn(pred,target,noisy)
            total_loss=total_loss+self.transient_weight*trans_loss

        return total_loss

In [19]:
import torch

def calculate_target (clean,noisy,predicted):
    clean=torch.nan_to_num(clean.float()).clamp(-1.0,1.0)
    noisy=torch.nan_to_num(noisy.float()).clamp(-1.0,1.0)
    predicted=torch.nan_to_num(predicted.float()).clamp(-1.0,1.0)

    original_noise=noisy-clean

    target_energy=torch.sum(clean**2,dim=-1)+1e-8
    error_energy=torch.sum((predicted-clean)**2,dim=-1)+1e-8
    original_noise_energy=torch.sum(original_noise**2,dim=-1)+1e-8

    distortion_ratio=torch.sqrt(error_energy/target_energy)
    quality_pct=(1.0-distortion_ratio)*100.0

    noise_reduction=1.0-(error_energy/original_noise_energy)
    noise_removal_pct=noise_reduction*100.0

    quality_score=torch.clamp(quality_pct,0.0,100.0).mean().item()
    noise_score=torch.clamp(noise_removal_pct,0.0,100.0).mean().item()

    return quality_score,noise_score

def calculate_transient_score(clean,noisy,predicted):
    clean=torch.nan_to_num(clean.float()).clamp(-1.0,1.0)
    noisy=torch.nan_to_num(noisy.float()).clamp(-1.0,1.0)
    predicted=torch.nan_to_num(predicted.float()).clamp(-1.0,1.0)

    noise_abs=(noisy-clean).abs()
    threshold=noise_abs.mean(dim=-1,keepdim=True)+2.5*noise_abs.std(dim=-1,keepdim=True)
    mask= (noise_abs>threshold).float()

    mask_sum=mask.sum(dim=-1,keepdim=True)
    mask=torch.where(mask_sum>10,mask,torch.ones_like(mask))

    before=(((noisy-clean)**2)*mask).sum(dim=-1)+1e-8
    after=(((predicted-clean)**2)*mask).sum(dim=-1)+1e-8
    score=(1.0-after/before)*100.0

    return torch.clamp(score,0.0,100.0).mean().item()

In [20]:
def calculate_snr(clean,test):

    clean=clean.float()
    test=test.float()

    signal_energy=torch.sum(clean**2,dim=-1)+1e-8
    noise_energy=torch.sum((test-clean)**2,dim=-1)+1e-8

    snr=10.0*torch.log10(signal_energy/noise_energy)

    return snr.mean().item()

def calculate_snr_metrics(clean,noisy,predicted):

    snr_before=calculate_snr(clean,noisy)
    snr_after=calculate_snr(clean,predicted)
    snr_improvement=snr_after-snr_before

    return snr_before,snr_after,snr_improvement

def calculate_si_sdr(clean,predicted):
    clean=clean.float()
    predicted=predicted.float()

    clean_zm=clean-clean.mean(dim=-1,keepdim=True)
    pred_zm=predicted-predicted.mean(dim=-1,keepdim=True)

    dot=(pred_zm*clean_zm).sum(dim=-1,keepdim=True)
    clean_energy=(clean_zm**2).sum(dim=-1,keepdim=True)+1e-8

    s_target=(dot/clean_energy)*clean_zm
    e_noise=pred_zm-s_target

    si_sdr=10.0*torch.log10(
        ((s_target**2).sum(dim=-1)+1e-8)/((e_noise**2).sum(dim=-1)+1e-8)
    )

    return si_sdr.mean().item()

def calculate_si_sdr_metrics(clean,noisy,predicted):

    si_sdr_before=calculate_si_sdr(clean,noisy)
    si_sdr_after=calculate_si_sdr(clean,predicted)
    si_sdr_improvement=si_sdr_after-si_sdr_before

    return si_sdr_before,si_sdr_after,si_sdr_improvement


In [21]:
@torch.no_grad()
def calculate_all_metrics(clean,noisy,predicted,pesq_fn=None,
                          stoi_fn=None,compute_pesq_stoi=False):
    clean=torch.nan_to_num(clean.float()).clamp(-1.0,1.0)
    noisy=torch.nan_to_num(noisy.float()).clamp(-1.0,1.0)
    predicted=torch.nan_to_num(predicted.float()).clamp(-1.0,1.0)

    quality,noise_rem=calculate_target(clean,noisy,predicted)
    transient=calculate_transient_score(clean,noisy,predicted)
    snr_before,snr_after,snr_improvement=calculate_snr_metrics(clean,noisy,predicted)
    si_sdr_before,si_sdr_after,si_sdr_improvement=calculate_si_sdr_metrics(clean,noisy,predicted)

    pesq_score=None
    stoi_score=None

    if compute_pesq_stoi:
        if pesq_fn:
            try:
                pesq_score=pesq_fn(predicted,clean).mean().item()
            except Exception:
                pesq_score=None
        if stoi_fn:
            try:
                stoi_score=stoi_fn(predicted,clean).mean().item()
            except Exception:
                stoi_score=None

    return {
        "quality":quality,
        "noise_rem":noise_rem,
        "transient":transient,

        "snr_before":snr_before,
        "snr_after":snr_after,
        "snr_improvement":snr_improvement,

        "si_sdr_before":si_sdr_before,
        "si_sdr_after":si_sdr_after,
        "si_sdr_improvement":si_sdr_improvement,

        "pesq":pesq_score,
        "stoi":stoi_score

    }


In [22]:
pesq_fn=PerceptualEvaluationSpeechQuality(16000,"wb").to(device)
stoi_fn=ShortTimeObjectiveIntelligibility(16000).to(device)

model=Model(
    base=48,
    gru_hidden_sz=512,
    layers=1,
    freq_bins=FREQ_BINS
).to(device)

transform=SpectrogramTransform(
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    center=CENTER,
    device=device
)
criterion=MasterHybridLoss().to(device)
optimizer=torch.optim.AdamW(model.parameters(),lr=0.0003,weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,mode="max",factor=0.5,patience=3,min_lr=1e-6
)
history={
    "train_loss":[],
    "val_loss":[],

    "quality":[],
    "noise_rem":[],
    "transient":[],

    "snr_before":[],
    "snr_after":[],
    "snr_improvement":[],

    "si_sdr_before":[],
    "si_sdr_after":[],
    "si_sdr_improvement":[],


    "pesq":[],
    "stoi":[]
}
best_score=-float("inf")
AMP=torch.cuda.is_available()
center=False
scaler=torch.cuda.amp.GradScaler(enabled=AMP)
total_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Streaming Denoiser is ready! Parameters : {total_params}")
print(f"AMP enabled: {AMP}")
print(f"Window Type : Hamming | center={center}")


Streaming Denoiser is ready! Parameters : 17909762
AMP enabled: True
Window Type : Hamming | center=False


/tmp/ipykernel_216902/2179829509.py:45: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler(enabled=AMP)


In [23]:
model.eval()
with torch.no_grad():
    dummy_wav=torch.randn(2,80000).to(device)
    dummy_spec=transform.audio_to_spectrogram(dummy_wav)
    dummy_mask,dummy_h=model(dummy_spec[:,:,:,:16])
    dummy_enh=transform.apply_mask(dummy_spec[:,:,:,:16],dummy_mask)
print("dummy_wav : ",dummy_wav.shape)
print("dummy_spec : ",dummy_spec.shape)
print("dummy_mask : ",dummy_mask.shape)
if dummy_h is not None:
    print("dummy_h : ",dummy_h.shape)
else:
    print("dummy_h : ",None)
with torch.no_grad():
    enhanced_wav,enhanced_spec,_=enhance_waveform_streaming(
        noisy_wav=dummy_wav,
        model=model,
        transform=transform,
        chunk_frames=TRAIN_CHUNK_FRAMES,
        h=None,
        detach_every=None
    )
print("enhanced_wav : ",enhanced_wav.shape)
print("enhanced_spec : ",enhanced_spec.shape)

assert enhanced_wav.shape==dummy_wav.shape
assert enhanced_spec.shape==dummy_spec.shape
print("Shape + ISTFT test passed.")

dummy_wav :  torch.Size([2, 80000])
dummy_spec :  torch.Size([2, 2, 256, 498])
dummy_mask :  torch.Size([2, 2, 256, 16])
dummy_h :  torch.Size([1, 2, 512])
enhanced_wav :  torch.Size([2, 80000])
enhanced_spec :  torch.Size([2, 2, 256, 498])
Shape + ISTFT test passed.


In [24]:
def unpack_batch(batch):
    if len(batch)==3:
        noisy_wav,clean_wav,names=batch
    else :
        noisy_wav,clean_wav=batch
        names=None
    return noisy_wav,clean_wav,names

def train_one_epoch(model,train_loader,optimizer,criterion,
                    transform,device,scaler=None,chunk_frames=TRAIN_CHUNK_FRAMES,
                    chunk_frame_choices=train_chunk_frame_choices,detach_every=DETACH_EVERY,
                    use_amp=True):
    model.train()
    total_loss=0.0
    num_batches=0
    pbar=tqdm(train_loader,desc="Training")

    for batch in pbar:
        noisy_wav,clean_wav,names=unpack_batch(batch)

        noisy_wav=noisy_wav.to(device,non_blocking=True).float()
        clean_wav=clean_wav.to(device,non_blocking=True).float()

        if chunk_frame_choices is not None and len(chunk_frame_choices)>0:
            cur_chunk_frame=random.choice(chunk_frame_choices)
        else :
            cur_chunk_frame=chunk_frames

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            enhanced_wav,enhanced_spec,_=enhance_waveform_streaming(
                noisy_wav=noisy_wav,
                model=model,
                transform=transform,
                chunk_frames=cur_chunk_frame,
                h=None,
                detach_every=detach_every
            )
            loss=criterion(enhanced_wav,clean_wav,noisy_wav)

        if scaler is not None and use_amp:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
        else :
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=5.0)
            optimizer.step()
        total_loss+=loss.item()
        num_batches+=1
        pbar.set_postfix({
            "loss":f"{loss.item():.4f}",
            "chunk":cur_chunk_frame
        })
    return total_loss/max(num_batches,1)


In [25]:
def validate_one_epoch(model,val_loader,criterion,transform,device,
                      pesq_fn=None,stoi_fn=None,chunk_frames=VAL_CHUNK_FRAMES,
                      compute_pesq_stoi=False,max_metric_batches=10):
    model.eval()
    total_loss=0.0
    total_metrics={}
    metric_counts={}
    num_batches=0

    pbar=tqdm(val_loader,desc="Validation")

    with torch.no_grad():
        for batch_idx,batch in enumerate(pbar):
            noisy_wav,clean_wav,names=unpack_batch(batch)

            noisy_wav=noisy_wav.to(device,non_blocking=True).float()
            clean_wav=clean_wav.to(device,non_blocking=True).float()
            enhanced_wav,enhanced_spec,_=enhance_waveform_streaming(
                noisy_wav=noisy_wav,
                model=model,
                transform=transform,
                chunk_frames=chunk_frames,
                h=None,
                detach_every=None
            )
            loss=criterion(enhanced_wav,clean_wav,noisy_wav)
            batch_metrics=calculate_all_metrics(
                clean=clean_wav,
                noisy=noisy_wav,
                predicted=enhanced_wav,
                pesq_fn=pesq_fn,
                stoi_fn=stoi_fn,
                compute_pesq_stoi=(
                    compute_pesq_stoi and batch_idx<max_metric_batches
                )
            )
            total_loss+=loss.item()
            for key,val in batch_metrics.items():
                if val is None:
                    continue
                total_metrics[key]=total_metrics.get(key,0.0)+float(val)
                metric_counts[key]=metric_counts.get(key,0)+1

            num_batches+=1

            pbar.set_postfix({
                "val_loss"      :f"{loss.item():.4f}",
                "Quality"       :f"{batch_metrics['quality']:.2f}",
                "Noise"         :f"{batch_metrics['noise_rem']:.2f}",
                "transient"     :f"{batch_metrics['transient']:.2f}",
                "snr_imp"       :f"{batch_metrics['snr_improvement']:.2f}",
                "si_sdr_imp"    :f"{batch_metrics['si_sdr_improvement']:.2f}"
            })
        denom=max(num_batches,1)
        avg_metrics={}
        for key,val in total_metrics.items():
            avg_metrics[key]=val/max(metric_counts.get(key,0),1)
        avg_metrics["val_loss"]=total_loss/denom
        avg_metrics.setdefault("pesq",0.0)
        avg_metrics.setdefault("stoi",0.0)
        return avg_metrics

In [ ]:
import math
os.makedirs("Final_sleep",exist_ok=True)
LAST_CKPT="Final_sleep/last_checkpoint.pth"
BEST_CKPT="Final_sleep/best_checkpoint.pth"
FINAL_MODEL="final_model_sleep.pth"
HISTORY_CSV="training_history_sleep.csv"

try:
    history
except NameError:
    history={
        "train_loss":[],
        "val_loss":[],

        "quality":[],
        "noise_rem":[],
        "transient":[],

        "snr_before":[],
        "snr_after":[],
        "snr_improvement":[],

        "si_sdr_before":[],
        "si_sdr_after":[],
        "si_sdr_improvement":[],

        "pesq":[],
        "stoi":[]
    }
def safe_torch_save(obj,path):
    tmp_path=path+".tmp"
    torch.save(obj,tmp_path)
    os.replace(tmp_path,path)
def safe_float(x,default=0.0):
    if x is None:
        return default
    try :
        x=float(x)
        if math.isnan(x) or math.isinf(x):
            return default
        return x
    except Exception:
        return default
start_epoch=0
try :
    best_score
except NameError:
    best_score=-float("inf")

if os.path.exists(LAST_CKPT):
    print(f"Loading checkpoint from : {LAST_CKPT}")
    ckpt=torch.load(LAST_CKPT,map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if "scheduler_state_dict" in ckpt and ckpt["scheduler_state_dict"] is not None:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    if "scaler_state_dict" in ckpt and ckpt["scaler_state_dict"] is not None:
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    start_epoch=ckpt["epoch"]+1
    best_score=ckpt.get("best_score",best_score)
    history=ckpt.get("history",history)
    print(f"Resuming from epoch {start_epoch+1}")
    print(f"Best score so far : {best_score:.4f}")
else:
    print("No checkpoint found. Starting from scratch. ")

Loading checkpoint from : Final_sleep/last_checkpoint.pth
Resuming from epoch 16
Best score so far : 5.5077


In [27]:
num_epochs=20
for epoch in range(start_epoch,num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    train_loss=train_one_epoch(
        model=model,
        train_loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        transform=transform,
        device=device,
        scaler=scaler,
        chunk_frames=TRAIN_CHUNK_FRAMES,
        chunk_frame_choices=train_chunk_frame_choices,
        detach_every=DETACH_EVERY,
        use_amp=True
    )
    compute_metrics=True
    val_metrics=validate_one_epoch(
        model=model,
        val_loader=val_loader,
        criterion=criterion,
        transform=transform,
        device=device,
        pesq_fn=pesq_fn,
        stoi_fn=stoi_fn,
        chunk_frames=VAL_CHUNK_FRAMES,
        compute_pesq_stoi=compute_metrics,
        max_metric_batches=10
    )

    val_loss=val_metrics["val_loss"]
    avg_quality=val_metrics["quality"]
    avg_noise=val_metrics["noise_rem"]
    avg_transient=val_metrics["transient"]

    avg_snr_before=val_metrics["snr_before"]
    avg_snr_after=val_metrics["snr_after"]
    avg_snr_improvement=val_metrics["snr_improvement"]

    avg_si_sdr_before=val_metrics["si_sdr_before"]
    avg_si_sdr_after=val_metrics["si_sdr_after"]
    avg_si_sdr_improvement=val_metrics["si_sdr_improvement"]


    avg_pesq=val_metrics["pesq"]
    avg_stoi=val_metrics["stoi"]

    scheduler.step(avg_pesq)

    history["train_loss"].append(train_loss)

    for key,val in val_metrics.items():
        if key in history:
            history[key].append(val)

    pd.DataFrame(history).to_csv("training_history_sleep.csv",index=False)

    score = (avg_pesq + 3*avg_stoi)

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Quality: {avg_quality:.4f}")
    print(f"Noise Removed: {avg_noise:.4f}")
    print(f"Transient: {avg_transient:.4f}")
    print()
    print(f"SNR Before: {avg_snr_before:.4f}")
    print(f"SNR After: {avg_snr_after:.4f}")
    print(f"SNR Improvement: {avg_snr_improvement:.4f}")
    print()
    print(f"SI-SDR Before: {avg_si_sdr_before:.4f}")
    print(f"SI-SDR After: {avg_si_sdr_after:.4f}")
    print(f"SI-SDR Improvement: {avg_si_sdr_improvement:.4f}")
    print()

    print(f"PESQ: {avg_pesq}")
    print(f"STOI: {avg_stoi}")
    print(f"Metrics: {compute_metrics}")


    checkpoint={
        "model_state_dict":model.state_dict(),
        "optimizer_state_dict":optimizer.state_dict(),
        "scheduler_state_dict":scheduler.state_dict() if scheduler is not None else None,
        "scaler_state_dict":scaler.state_dict() if scaler is not None else None,
        "epoch":epoch,
        "best_score":best_score,
        "current_score":score,
        "history":history,
        "last_val_metrics":val_metrics,

        "config":{
                    "base":model.base,
                    "gru_hidden_sz":model.gru_hidden_sz,
                    "layers":model.layers,
                    "freq_bins":model.freq_bins,
                    "n_fft":N_FFT,
                    "hop_length":HOP_LENGTH,
                    "center":CENTER,
                    "window":"hamming",
                    "train_chunk_frame":TRAIN_CHUNK_FRAMES,
                    "train_chunk_frame_choices":train_chunk_frame_choices,
                    "val_chunk_frames":VAL_CHUNK_FRAMES,
                    "real_time_chunk_frames":REAL_TIME_CHUNK_FRAMES,
                    "enable_transient_aug":True,
                    "transient_prob":TRANSIENT_PROB,
                    "transient_loss_weight":TRANSIENT_LOSS_WEIGHT,
                    "best_score_formula":("(avg_pesq + 3*avg_stoi)")

                }

    }



    if compute_metrics and score>best_score:
        best_score=score
        checkpoint["best_score"]=best_score
        safe_torch_save(checkpoint,BEST_CKPT)
        safe_torch_save(checkpoint,FINAL_MODEL)

        print(f"Saved best checkpoint to : {BEST_CKPT}")
        print(f"Saved final model to : {FINAL_MODEL}")

    checkpoint["best_score"]=best_score
    safe_torch_save(checkpoint,LAST_CKPT)
    print(f"Saved last checkpoint to : {LAST_CKPT}")
    print("-"*60)
print("\nTraining finished.")
print(f"Best score: {best_score:.4f}")
print(f"Last checkpoint: {LAST_CKPT}")
print(f"Best checkpoint: {BEST_CKPT}")
print(f"Final model: {FINAL_MODEL}")


Epoch 16/20


Training:   0%|          | 0/2528 [00:00<?, ?it/s]/tmp/ipykernel_216902/1498649009.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Validation:  25%|██▌       | 295/1157 [02:11<04:40,  3.07it/s, val_loss=-25.5131, Quality=89.49, Noise=82.43, transient=88.63, snr_imp=8.63, si_sdr_imp=8.61]  

Error: Error opening '/teamspace/studios/this_studio/MeetVerse_DataSet/Clear Sound val/VS-11-FA-300/VS-11-FA-235.mp3': File does not exist or is not a regular file (possibly a pipe?).


[src/libmpg123/parse.c:do_readahead():1083] warning: Cannot read next header, a one-frame stream? Duh...
Validation: 100%|██████████| 1157/1157 [06:53<00:00,  2.79it/s, val_loss=-19.8655, Quality=80.67, Noise=76.40, transient=85.43, snr_imp=5.64, si_sdr_imp=5.35]  


...

Train Loss: -21.6860
Val Loss: -21.5430
Quality: 85.5451
Noise Removed: 79.0148
Transient: 86.3976

SNR Before: 14.0909
SNR After: 22.3457
SNR Improvement: 8.2547

SI-SDR Before: 14.0955
SI-SDR After: 22.2161
SI-SDR Improvement: 8.1205

PESQ: 2.6232168912887572
STOI: 0.918152779340744
Metrics: True
Saved last checkpoint to : Final_sleep/last_checkpoint.pth
------------------------------------------------------------

Epoch 20/20


Validation:  25%|██▌       | 295/1157 [02:07<04:41,  3.06it/s, val_loss=-18.3270, Quality=87.25, Noise=84.33, transient=91.31, snr_imp=9.31, si_sdr_imp=9.23]  

Error: Error opening '/teamspace/studios/this_studio/MeetVerse_DataSet/Clear Sound val/VS-11-FA-300/VS-11-FA-235.mp3': File does not exist or is not a regular file (possibly a pipe?).


[src/libmpg123/parse.c:do_readahead():1083] warning: Cannot read next header, a one-frame stream? Duh...
Validation: 100%|██████████| 1157/1157 [06:50<00:00,  2.82it/s, val_loss=-15.0123, Quality=78.79, Noise=85.27, transient=93.24, snr_imp=9.41, si_sdr_imp=9.17]  


Train Loss: -21.6439
Val Loss: -21.5525
Quality: 85.6466
Noise Removed: 78.8362
Transient: 86.6870

SNR Before: 14.2055
SNR After: 22.3319
SNR Improvement: 8.1264

SI-SDR Before: 14.2110
SI-SDR After: 22.2161
SI-SDR Improvement: 8.0051

PESQ: 2.709130334854126
STOI: 0.917617654800415
Metrics: True
Saved last checkpoint to : Final_sleep/last_checkpoint.pth
------------------------------------------------------------

Training finished.
Best score: 5.5178
Last checkpoint: Final_sleep/last_checkpoint.pth
Best checkpoint: Final_sleep/best_checkpoint.pth
Final model: final_model_sleep.pth


In [28]:
# mkdir -p logs nohvert --execute --inplace --ExecutePreprocessor.timeout=-1 "Final_sleep.ipynb" > logs/run.log 2>&1 &